In [ ]:
# ============================================
# 📦 Install Required Libraries (if needed)
# ============================================
# Uncomment if running first time in Colab
# !pip install pandas numpy scikit-learn

# ============================================
# 📚 Import Libraries
# ============================================

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [12]:
!rm -rf ~/.kaggle
!rm kaggle.json

In [25]:
# COMPLETE Kaggle setup in ONE cell

import json
import os

# Step 1: Create kaggle.json
kaggle_dict = {
    "username": "ranganathrangam",
    "key": "PASTE_YOUR_NEW_API_KEY"
}

with open("kaggle.json", "w") as f:
    json.dump(kaggle_dict, f)

# Step 2: Setup directory properly
os.system("rm -rf ~/.kaggle")
os.system("mkdir -p ~/.kaggle")
os.system("cp kaggle.json ~/.kaggle/")
os.system("chmod 600 ~/.kaggle/kaggle.json")

# Step 3: Test Kaggle
print("Checking Kaggle connection...")
os.system("kaggle datasets list")

# Step 4: Download dataset
print("Downloading dataset...")
os.system("kaggle datasets download -d retailrocket/ecommerce-dataset")
os.system("unzip ecommerce-dataset.zip")

print("✅ DONE")

Checking Kaggle connection...
✅ DONE


In [26]:
!ls

 category_tree.csv	     item_properties_part2.csv	 sample_data
 ecommerce-dataset.zip	    'kaggle (1).json'		'Untitled0 (1).ipynb'
 events.csv		    'kaggle (2).json'
 item_properties_part1.csv   kaggle.json


In [27]:
!unzip -o ecommerce-dataset.zip

Archive:  ecommerce-dataset.zip
  inflating: category_tree.csv       
  inflating: events.csv              
  inflating: item_properties_part1.csv  
  inflating: item_properties_part2.csv  


In [17]:
# ============================================
# 📊 Load Dataset
# ============================================

df = pd.read_csv("events.csv")

# Take sample for faster training (optional)
df = df.sample(n=200000, random_state=42)

# ============================================
# 👀 Inspect Data
# ============================================

print(df.head())
print(df.info())

             timestamp  visitorid event  itemid  transactionid
486798   1435193216976      50734  view    4442            NaN
1145255  1440996903983     355903  view  269631            NaN
1601366  1431280237515    1066758  view  221329            NaN
843976   1439507864399    1049477  view   23683            NaN
2524686  1437601159324     143239  view    6552            NaN
<class 'pandas.core.frame.DataFrame'>
Index: 200000 entries, 486798 to 662795
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   timestamp      200000 non-null  int64  
 1   visitorid      200000 non-null  int64  
 2   event          200000 non-null  object 
 3   itemid         200000 non-null  int64  
 4   transactionid  1616 non-null    float64
dtypes: float64(1), int64(3), object(1)
memory usage: 9.2+ MB
None


In [28]:
#Remove Duplicates
df.drop_duplicates(inplace=True)

# Dataset timestamp is in milliseconds
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')


In [29]:
#Check Event Types
df['event'].value_counts()

,count
event,
view,193281
addtocart,5100
transaction,1616


In [31]:
#Convert Event to Numeric
df['event_type'] = df['event'].map({
    'view': 0,
    'addtocart': 1,
    'transaction': 2
})
#Create Target Variable
df['purchased'] = df['event'].apply(lambda x: 1 if x == 'transaction' else 0)

In [32]:
#Extract Time Features
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.dayofweek
#quick check
df.head()

,timestamp,visitorid,event,itemid,transactionid,is_purchase,hour,day,event_type,purchased
486798,2015-06-25 00:46:56.976,50734,view,4442,NaN,0,0,3,0,0
1145255,2015-08-31 04:55:03.983,355903,view,269631,NaN,0,4,0,0,0
1601366,2015-05-10 17:50:37.515,1066758,view,221329,NaN,0,17,6,0,0
843976,2015-08-13 23:17:44.399,1049477,view,23683,NaN,0,23,3,0,0
2524686,2015-07-22 21:39:19.324,143239,view,6552,NaN,0,21,2,0,0


In [34]:


# Load only part of dataset to speed up training
df = pd.read_csv("events.csv", nrows=200000)

print("Data loaded:", df.shape)


# ============================================
# 🧹 STEP 3: FILTER ACTIVE USERS (VERY IMPORTANT)
# ============================================

# Keep users with more than 3 interactions
user_counts = df['visitorid'].value_counts()
active_users = user_counts[user_counts > 3].index

df = df[df['visitorid'].isin(active_users)]

print("After filtering active users:", df.shape)

Data loaded: (200000, 5)
After filtering active users: (73318, 5)


In [36]:
#TIME FEATURES
# ============================================

df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.dayofweek

In [37]:
# 🔢 STEP 5: EVENT ENCODING
# ============================================

event_map = {
    'view': 0,
    'addtocart': 1,
    'transaction': 2
}

df['event_score'] = df['event'].map(event_map)


In [38]:
# Target variable
df['purchased'] = (df['event'] == 'transaction').astype(int)

In [42]:
#TRAIN-TEST SPLIT
# ============================================

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


# ============================================
# 🧠 STEP 7: FEATURE ENGINEERING FUNCTION
# ============================================

def create_features(data):

    # -------- USER FEATURES --------
    user_features = data.groupby('visitorid').agg({
        'hour': lambda x: x.mode()[0],
        'day': lambda x: x.mode()[0],
        'event_score': 'mean',
        'event': 'count'
    }).rename(columns={
        'event': 'total_events',
        'event_score': 'avg_event_score'
    }).reset_index()

    # Purchase count (strong signal)
    purchase_count = data[data['event'] == 'transaction'].groupby('visitorid').size()
    user_features['purchase_count'] = user_features['visitorid'].map(purchase_count).fillna(0)

    # -------- EVENT COUNTS --------
    event_counts = data.pivot_table(
        index='visitorid',
        columns='event',
        aggfunc='size',
        fill_value=0
    ).reset_index()

    event_counts.columns.name = None

    user_features = user_features.merge(event_counts, on='visitorid', how='left')

    # -------- BEHAVIOR FEATURES --------
    user_features['view_to_cart_ratio'] = user_features.get('addtocart', 0) / (user_features.get('view', 0) + 1)
    user_features['has_cart_activity'] = (user_features.get('addtocart', 0) > 0).astype(int)

    # -------- ITEM FEATURES --------
    item_features = data.groupby('itemid').agg({
        'event': 'count',
        'event_score': 'mean'
    }).rename(columns={
        'event': 'item_popularity',
        'event_score': 'item_engagement'
    }).reset_index()

    return user_features, item_features


# ============================================
# 🏗️ STEP 8: BUILD FEATURES (TRAIN + TEST)
# ============================================

user_features_train, item_features_train = create_features(train_df)
user_features_test, item_features_test = create_features(test_df)


# ============================================
# 🔗 STEP 9: MERGE FEATURES
# ============================================

train_data = train_df.merge(user_features_train, on='visitorid')
train_data = train_data.merge(item_features_train, on='itemid')

test_data = test_df.merge(user_features_test, on='visitorid')
test_data = test_data.merge(item_features_test, on='itemid')

train_data.rename(columns={
    'hour_y': 'hour',
    'day_y': 'day'
}, inplace=True)

test_data.rename(columns={
    'hour_y': 'hour',
    'day_y': 'day'
}, inplace=True)


# ============================================
# 📌 STEP 10: SELECT FEATURES
# ============================================

features = [
    'hour',
    'day',
    'total_events',
    'avg_event_score',
    'purchase_count',
    'view',
    'addtocart',
    'transaction',
    'view_to_cart_ratio',
    'has_cart_activity',
    'item_popularity',
    'item_engagement'
]

X_train = train_data[features].fillna(0)
y_train = train_data['purchased']

X_test = test_data[features].fillna(0)
y_test = test_data['purchased']


# ============================================
# 🤖 STEP 11: TRAIN MODEL (FASTER SETTINGS)
# ============================================

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=10,  # IMPORTANT for imbalance
    random_state=42
)

model.fit(X_train, y_train)
# ============================================
# 🔍 CHECK 1: TRAIN vs TEST PERFORMANCE
# ============================================

# Train performance
y_train_pred = model.predict(X_train)
print("Train Performance:\n")
print(classification_report(y_train, y_train_pred))

# Test performance
y_test_pred = model.predict(X_test)
print("\nTest Performance:\n")
print(classification_report(y_test, y_test_pred))


# ============================================
# 📊 STEP 12: EVALUATE MODEL
# ============================================

y_prob = model.predict_proba(X_test)[:, 1]

y_pred = (y_prob > 0.5).astype(int)

print("\nModel Performance:\n")
print(classification_report(y_test, y_pred))


# ============================================
# 💾 STEP 13: SAVE MODEL
# ============================================

pickle.dump(model, open("rf_model_v1.pkl", "wb"))


pickle.dump(features, open("features_v1.pkl", "wb"))

print("\n✅ Model saved successfully!")
print("\n features saved successfully")

Train Performance:

              precision    recall  f1-score   support

           0       1.00      0.94      0.97     57555
           1       0.23      0.95      0.38      1099

    accuracy                           0.94     58654
   macro avg       0.62      0.95      0.67     58654
weighted avg       0.98      0.94      0.96     58654


Test Performance:

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     14419
           1       0.49      1.00      0.66       245

    accuracy                           0.98     14664
   macro avg       0.74      0.99      0.82     14664
weighted avg       0.99      0.98      0.99     14664


Model Performance:

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     14419
           1       0.49      1.00      0.66       245

    accuracy                           0.98     14664
   macro avg       0.74      0.99      0.82     14664
weighted avg 